# 🔬 Reaction-Diffusion Theory of Microvascular Pharmacological Sanctuaries and Inflammatory Pre-conditioning
### Interactive Biophysical Solvers, Volterra Memory Functional & Clinical Survival Analysis (N = 3,461)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lukecele/inflammatory-onco-physics/blob/main/demo_inflammatory_onco_physics.ipynb)
[![Colab Pro: T4 GPU](https://img.shields.io/badge/Colab_Pro-T4_GPU_Accelerated-orange.svg?logo=googlecolab)](https://colab.research.google.com/github/Lukecele/inflammatory-onco-physics/blob/main/demo_inflammatory_onco_physics.ipynb)
[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.24118328.svg)](https://doi.org/10.5281/zenodo.24118328)
[![License: CC BY 4.0](https://img.shields.io/badge/License-CC_BY_4.0-lightgrey.svg)](https://creativecommons.org/licenses/by/4.0/)
[![REMARK Compliant](https://img.shields.io/badge/REMARK-Compliant-success.svg)](https://github.com/Lukecele/inflammatory-onco-physics/blob/main/REMARK_COMPLIANCE_REPORT.md)

**Author:** Luca Celebrano<sup>1</sup>  
<sup>1</sup> Department of Clinical Medicine and Surgery, School of Medicine and Surgery, Università degli Studi di Napoli Federico II, Naples, Italy  
*Correspondence:* [l.celebrano@studenti.unina.it](mailto:l.celebrano@studenti.unina.it) | ORCID: [0009-0007-0931-3555](https://orcid.org/0009-0007-0931-3555)  
**GitHub Repository:** [Lukecele/inflammatory-onco-physics](https://github.com/Lukecele/inflammatory-onco-physics)

---
**Author:** Luca Celebrano<sup>1</sup>  
<sup>1</sup> Department of Clinical Medicine and Surgery, School of Medicine and Surgery, Università degli Studi di Napoli Federico II, Naples, Italy  
*Correspondence:* [l.celebrano@studenti.unina.it](mailto:l.celebrano@studenti.unina.it) | ORCID: [0009-0007-0931-3555](https://orcid.org/0009-0007-0931-3555)  
**GitHub Repository:** [Lukecele/inflammatory-onco-physics](https://github.com/Lukecele/inflammatory-onco-physics)

---

### 📌 Theoretical Foundations
This interactive notebook reproduces the continuous biophysical models and empirical survival validations presented in:
> **Celebrano, L. (2026).** *A Reaction-Diffusion Theory of Microvascular Pharmacological Sanctuaries and Inflammatory Pre-conditioning Explains Post-Adjuvant Relapse Dynamics Across 3,461 Patients.* Springer Nature Submission / Zenodo Archive. [DOI: 10.5281/zenodo.24118328](https://doi.org/10.5281/zenodo.24118328).

#### 1. Cylindrical Bessel $K_0$ Reaction-Diffusion
In desmoplastic microenvironments ($Pe < 0.1$), transport from patent microvessels ($r_v = 10\ \mu\text{m}$) follows radial diffusion with metabolic degradation ($k = 1.51 \times 10^{-4}\text{ s}^{-1}$):
$$C(r) = C_0 \frac{K_0(r / \lambda)}{K_0(r_v / \lambda)}, \quad \lambda = \sqrt{\frac{D_{\text{eff}}}{k}} \approx 65.57\ \mu\text{m}$$
At $r_{\text{crit}} \approx 165.89\ \mu\text{m}$, drug levels fall below $\text{IC}_{50} = 10\text{ nM}$, while oxygen supports cellular survival up to $r_{\text{ox}} \approx 200\ \mu\text{m}$. A **34.1 $\mu$m perivascular cytostatic sanctuary sleeve (31.28% of viable tumor cord volume)** emerges.

#### 2. Volterra Memory Functional & Double-Barrier ctDNA Shedding
Dormant persister (DTP) cells awaken via systemic host inflammation (IL-1β, IL-6, TNF-α, neutrophil NETosis), modeled by the hereditary exhaustion functional $\mathcal{E}_{\text{imm}}(t) = \mathcal{E}_0 \exp(-\gamma \int_0^t \Phi(x,\tau)d\tau)$. Steric collagen entrapment (pore size $\le 11\text{ nm}$) causes early molecular residual disease (MRD) false negativity, followed by an analytical ctDNA lead time of **5.46 months** before radiological recurrence.

In [ ]:
# Setup environment, auto-clone on Colab, and inspect hardware acceleration (GPU T4)
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    print("[*] Running on Google Colab (Pro). Cloning repository...")
    !git clone https://github.com/Lukecele/inflammatory-onco-physics.git
    %cd inflammatory-onco-physics
    !pip install -q lifelines

import numpy as np
import pandas as pd
import scipy.special as sp
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test

# Check PyTorch & T4 GPU availability
try:
    import torch
    if torch.cuda.is_available():
        dev_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"🚀 Hardware Acceleration Active: NVIDIA {dev_name} ({vram_gb:.2f} GB VRAM)")
    else:
        print("ℹ️ Standard CPU Runtime active (In Colab Pro, select Runtime -> Change runtime type -> T4 GPU for acceleration).")
except ImportError:
    pass

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.family"] = "sans-serif"
print("✅ Environment initialized successfully.")


In [ ]:
# 1. CYLINDRICAL BESSEL K0 SANCTUARY SOLVER
r_v = 10.0      # Capillary vessel radius (microns)
r_ox = 200.0    # Outer tumor cord boundary / normoxic margin (microns)
C_0 = 100.0     # Microvessel luminal concentration (nM)
IC_50 = 10.0    # Cytocidal concentration threshold (nM)
D_eff = 650.0   # Effective interstitial diffusion coefficient (um^2/s)
k_deg = 1.51e-4 # Tissue metabolic clearance (s^-1)
lam_C = np.sqrt(D_eff / k_deg) # Penetration length (65.57 um)

r = np.linspace(r_v, r_ox, 1000)
C_r = C_0 * sp.kn(0, r / lam_C) / sp.kn(0, r_v / lam_C)

idx_crit = np.where(C_r <= IC_50)[0][0]
r_crit = r[idx_crit]

V_total = np.pi * (r_ox**2 - r_v**2)
V_sanctuary = np.pi * (r_ox**2 - r_crit**2)
frac_sanctuary = (V_sanctuary / V_total) * 100

print(f'[*] Penetration depth lambda:       {lam_C:.2f} um')
print(f'[*] Critical cytocidal boundary r:  {r_crit:.2f} um')
print(f'[*] Distal sanctuary sleeve width:  {r_ox - r_crit:.2f} um')
print(f'[*] Sanctuary Volume Fraction:      {frac_sanctuary:.2f}%')

fig, ax = plt.subplots(figsize=(9, 4.5), dpi=150)
ax.plot(r, C_r, color='#0284c7', lw=2.5, label=r'Osimertinib Concentration $C(r)$')
ax.axhline(IC_50, color='#dc2626', ls='--', lw=1.8, label=r'Cytocidal Threshold ($\mathrm{IC}_{50} = 10\,\mathrm{nM}$)')
ax.axvline(r_crit, color='#d97706', ls=':', lw=1.8, label=f'Critical Boundary ($r_{{\\mathrm{{crit}}}} = {r_crit:.1f}\,\\mu\\mathrm{{m}}$)')
ax.fill_between(r[r >= r_crit], 0, C_r[r >= r_crit], color='#fef3c7', alpha=0.85, label=f'DTP Sanctuary Sleeve ({frac_sanctuary:.1f}% Volume)')
ax.set_xlabel(r'Radial Distance from Vessel Center $r\ (\mu\mathrm{m})$', fontsize=11, fontweight='bold')
ax.set_ylabel('Free Drug Concentration (nM)', fontsize=11, fontweight='bold')
ax.set_title(r'Cylindrical Bessel $K_0$ Radial Diffusion & Pharmacological Sanctuary', fontsize=12, fontweight='bold', pad=12)
ax.set_xlim(r_v, r_ox)
ax.set_ylim(0, 105)
ax.legend(frameon=True, facecolor='white', framealpha=0.95, loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 2. DOUBLE-BARRIER ctDNA SHEDDING & ANALYTICAL LEAD TIME
T_d = 24.6 # Doubling time (days)
t_days = np.linspace(0, 365 * 1.5, 500)
N_0 = 1e4  # Residual persister burden at drug cessation
N_t = N_0 * (2.0 ** (t_days / T_d))

# Steric collagen entrapment: shedding increases with tumor-induced matrix remodeling
h_shed = 1e-4 * (1.0 + np.tanh((np.log10(N_t) - 6.0) * 1.5))
ctDNA_vaf = h_shed * (N_t / 1e9) * 100 # VAF (%)

limit_detection_ctdna = 0.02 # High-sensitivity digital NGS limit (0.02% VAF)
limit_detection_imaging = 1e8 # CT macro-lesion (~10^8 cells, ~5-8 mm)

idx_ctdna = np.where(ctDNA_vaf >= limit_detection_ctdna)[0][0]
idx_imaging = np.where(N_t >= limit_detection_imaging)[0][0]
t_ctdna = t_days[idx_ctdna]
t_imaging = t_days[idx_imaging]
lead_time_months = (t_imaging - t_ctdna) / 30.4375

print(f'[*] ctDNA Molecular Positivity:  Day {t_ctdna:.1f} ({t_ctdna/30.4375:.1f} months)')
print(f'[*] Radiological Macro-Recurrence: Day {t_imaging:.1f} ({t_imaging/30.4375:.1f} months)')
print(f'[*] Analytical ctDNA Lead Time:    {lead_time_months:.2f} months')

fig, ax1 = plt.subplots(figsize=(9, 4.5), dpi=150)
color = '#1e293b'
ax1.set_xlabel('Time Post-Adjuvant Discontinuation (Days)', fontsize=11, fontweight='bold')
ax1.set_ylabel(r'Viable Tumor Cell Burden $N(t)$ (Log Scale)', color=color, fontsize=11, fontweight='bold')
ax1.semilogy(t_days, N_t, color=color, lw=2.2, label='Cellular Burden $N(t)$')
ax1.axhline(limit_detection_imaging, color='#dc2626', ls='--', lw=1.5, label='CT Radiological Detection ($10^8$ cells)')
ax1.axvline(t_imaging, color='#dc2626', ls=':', alpha=0.7)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = '#0284c7'
ax2.set_ylabel('Plasma ctDNA Variant Allele Frequency VAF (%)', color=color, fontsize=11, fontweight='bold')
ax2.plot(t_days, ctDNA_vaf, color=color, lw=2.5, ls='-', label='Plasma ctDNA VAF (%)')
ax2.axhline(limit_detection_ctdna, color='#059669', ls='--', lw=1.5, label='Liquid Biopsy MRD Threshold (0.02%)')
ax2.axvline(t_ctdna, color='#059669', ls=':', alpha=0.7)
ax2.tick_params(axis='y', labelcolor=color)

ax1.axvspan(t_ctdna, t_imaging, color='#10b981', alpha=0.15, label=f'Analytical Lead Time Window ({lead_time_months:.1f} Months)')
fig.suptitle('Double-Barrier ctDNA Shedding Dynamics & Pre-Radiological Lead Time', fontsize=12, fontweight='bold')
ax1.legend(loc='upper left', fontsize=9, framealpha=0.9)
ax2.legend(loc='center left', fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:
# 3. EMPIRICAL CLINICAL VALIDATION (Real Blood Biomarkers N = 1,056)
data_path = Path('data/real_gastric_nlr_crp_dataset.csv')
if not data_path.exists():
    data_path = Path('../data/real_gastric_nlr_crp_dataset.csv')

df_blood = pd.read_csv(data_path)
df_clean = df_blood.dropna(subset=['OS_MONTHS', 'DEATH_EVENT', 'RAW_PREOP_NLR', 'GPS']).copy()
df_clean['HTBI_SCORE'] = (np.log1p(df_clean['RAW_PREOP_NLR']) + df_clean['GPS']) / 4.0
median_cut = df_clean['HTBI_SCORE'].median()
df_clean['HTBI_HIGH'] = (df_clean['HTBI_SCORE'] >= median_cut).astype(int)

print(f'[*] Audited Cohort Size:       N = {len(df_clean)} complete patient records')
print(f'[*] Observed Death Events:     {int(df_clean["DEATH_EVENT"].sum())} deaths')
print(f'[*] Pre-Op NLR Median:         {df_clean["RAW_PREOP_NLR"].median():.2f}')

km_low = KaplanMeierFitter()
km_high = KaplanMeierFitter()
mask_high = df_clean['HTBI_HIGH'] == 1

km_low.fit(df_clean.loc[~mask_high, 'OS_MONTHS'], df_clean.loc[~mask_high, 'DEATH_EVENT'], label='Low Systemic Inflammation (HTBI Low)')
km_high.fit(df_clean.loc[mask_high, 'OS_MONTHS'], df_clean.loc[mask_high, 'DEATH_EVENT'], label='High Systemic Inflammation (HTBI High)')

lr = logrank_test(df_clean.loc[~mask_high, 'OS_MONTHS'], df_clean.loc[mask_high, 'OS_MONTHS'],
                  df_clean.loc[~mask_high, 'DEATH_EVENT'], df_clean.loc[mask_high, 'DEATH_EVENT'])

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)
km_low.plot_survival_function(ax=ax, color='#0284c7', lw=2.2)
km_high.plot_survival_function(ax=ax, color='#dc2626', lw=2.2)
ax.set_title(f'Real Surgical Blood Biomarker Cohort (N = {len(df_clean)})\nEmpirical Overall Survival by Baseline Inflammation (Log-Rank p = {lr.p_value:.2e})', fontsize=11, fontweight='bold', pad=10)
ax.set_xlabel('Overall Survival Time (Months)', fontsize=10, fontweight='bold')
ax.set_ylabel('Overall Survival Probability', fontsize=10, fontweight='bold')
ax.set_ylim(0.4, 1.02)
ax.legend(frameon=True, facecolor='white', fontsize=9)
plt.tight_layout()
plt.show()

cph = CoxPHFitter()
cox_df = df_clean[['OS_MONTHS', 'DEATH_EVENT', 'HTBI_HIGH', 'TNM_STAGE']].dropna()
cph.fit(cox_df, duration_col='OS_MONTHS', event_col='DEATH_EVENT')
hr = cph.hazard_ratios_['HTBI_HIGH']
ci = np.exp(cph.confidence_intervals_.loc['HTBI_HIGH'])
p_val = cph.summary.loc['HTBI_HIGH', 'p']
print(f'\n[✔] Multivariable Cox Regression (Controlling for Pathological TNM Stage):')
print(f'    • HTBI Hazard Ratio:  HR = {hr:.3f} (95% CI: {ci.iloc[0]:.3f} - {ci.iloc[1]:.3f})')
print(f'    • Statistical Signif: p = {p_val:.4e}')

---
### 📬 Citation & Contact
```bibtex
@article{celebrano2026inflammatoryoncophysics,
  title={A Reaction-Diffusion Theory of Microvascular Pharmacological Sanctuaries and Inflammatory Pre-conditioning Explains Post-Adjuvant Relapse Dynamics Across 3,461 Patients},
  author={Celebrano, Luca},
  journal={Zenodo Archive / Springer Nature Submission},
  year={2026},
  doi={10.5281/zenodo.24118328},
  url={https://doi.org/10.5281/zenodo.24118328},
}
```
*Correspondence:* Luca Celebrano (`l.celebrano@studenti.unina.it` / `luca.celebrano@gmail.com`)